# Exploring weekly dengue reports in Sri Lanka

My first task is to identify the available dates, district names and case
counts before choosing the study period.

Data source: denguedatahub, srilanka_weekly_data.
Original source: Weekly Epidemiological Reports, Ministry of Health, Sri Lanka.
AI assistance: ChatGPT helped draft the initial data-loading snippet.

In [1]:
import pandas as pd

data_url = (
    "https://thiyangt.r-universe.dev/denguedatahub/"
    "data/srilanka_weekly_data/csv"
)

dengue = pd.read_csv(data_url)

print("Rows and columns:", dengue.shape)
print("Column names:", dengue.columns.tolist())
display(dengue.head())

Rows and columns: (26753, 6)
Column names: ['year', 'week', 'start.date', 'end.date', 'district', 'cases']


,year,week,start.date,end.date,district,cases
0,2006,52,12/23/2006,12/29/2006,Colombo,71
1,2006,52,12/23/2006,12/29/2006,Gampaha,12
2,2006,52,12/23/2006,12/29/2006,Kalutara,12
3,2006,52,12/23/2006,12/29/2006,Kandy,20
4,2006,52,12/23/2006,12/29/2006,Matale,4


In [3]:
import pandas as pd

weekly = dengue.copy()
weekly["district"] = weekly["district"].str.strip()

def parse_report_date(column):
    slash_dates = pd.to_datetime(column, format="%m/%d/%Y", errors="coerce")
    iso_dates = pd.to_datetime(column, format="%Y-%m-%d", errors="coerce")
    return slash_dates.fillna(iso_dates)

weekly["week_start"] = parse_report_date(weekly["start.date"])
weekly["week_end"] = parse_report_date(weekly["end.date"])

print("Unreadable start dates:", weekly["week_start"].isna().sum())
print("Unreadable end dates:", weekly["week_end"].isna().sum())
print("First reporting week starts:", weekly["week_start"].min().date())
print("Last reporting week starts:", weekly["week_start"].max().date())
print("Last reporting week ends:", weekly["week_end"].max().date())

Unreadable start dates: 0
Unreadable end dates: 0
First reporting week starts: 2006-12-23
Last reporting week starts: 2026-09-07
Last reporting week ends: 2026-09-13


In [4]:
# One row per reporting week, one column per original reporting area.
area_cases = weekly.pivot(
    index="week_start",
    columns="district",
    values="cases"
).sort_index()

# Combine the two Ampara reporting areas into one administrative district.
district_cases = area_cases.copy()
district_cases["Ampara"] = area_cases["Ampara"] + area_cases["Kalmune"]
district_cases = district_cases.drop(columns="Kalmune")

# Standardise two labels for display in the dashboard.
district_cases = district_cases.rename(columns={
    "Hambanthota": "Hambantota",
    "NuwaraEliya": "Nuwara Eliya"
}).sort_index(axis="columns")

print("Original reporting areas:", area_cases.shape[1])
print("Reporting weeks and districts:", district_cases.shape)
display(district_cases.head())

Original reporting areas: 26
Reporting weeks and districts: (1029, 25)


district,Ampara,Anuradhapura,Badulla,Batticaloa,Colombo,Galle,Gampaha,Hambantota,Jaffna,Kalutara,...,Matale,Matara,Monaragala,Mullaitivu,Nuwara Eliya,Polonnaruwa,Puttalam,Ratnapura,Trincomalee,Vavuniya
week_start,,,,,,,,,,,,,,,,,,,,,
2006-12-23,0.0,4.0,2.0,0.0,71.0,1.0,12.0,1.0,0.0,12.0,...,4.0,11.0,0.0,0.0,1.0,1.0,6.0,1.0,0.0,3.0
2006-12-30,0.0,0.0,0.0,0.0,40.0,10.0,11.0,1.0,0.0,20.0,...,2.0,8.0,0.0,0.0,4.0,4.0,13.0,5.0,2.0,3.0
2007-01-06,0.0,0.0,2.0,0.0,43.0,4.0,13.0,2.0,0.0,7.0,...,7.0,8.0,1.0,0.0,2.0,0.0,9.0,3.0,1.0,0.0
2007-01-13,0.0,0.0,2.0,0.0,38.0,4.0,16.0,1.0,0.0,6.0,...,7.0,4.0,1.0,0.0,2.0,2.0,11.0,2.0,2.0,0.0
2007-01-20,0.0,2.0,1.0,0.0,52.0,3.0,12.0,1.0,0.0,15.0,...,4.0,4.0,3.0,0.0,3.0,2.0,8.0,2.0,2.0,2.0


In [5]:
missing_records = (
    district_cases.isna()
    .rename_axis(index="week_start", columns="district")
    .stack()
)

print("Missing district-week records:", missing_records.sum())
print(missing_records[missing_records].index.tolist())

Missing district-week records: 1
[(Timestamp('2026-02-09 00:00:00'), 'Puttalam')]


#Baseline forecast and evaluation

In [6]:
district_cases = district_cases.sort_index()
weeks = district_cases.index

train_weeks = weeks[:-104]        # Earlier weeks for training future models
validation_weeks = weeks[-104:-52]  # Next 52 weeks for comparing approaches
test_weeks = weeks[-52:]          # Final 52 weeks for the final evaluation

for name, dates in [
    ("Training", train_weeks),
    ("Validation", validation_weeks),
    ("Test", test_weeks),
]:
    print(f"{name}: {len(dates)} weeks | {dates.min().date()} to {dates.max().date()}")

Training: 925 weeks | 2006-12-23 to 2024-09-07
Validation: 52 weeks | 2024-09-14 to 2025-09-06
Test: 52 weeks | 2025-09-13 to 2026-09-07


In [7]:
# Move each district's observed count forward by exactly seven days.
# Keep only the target weeks we want to evaluate.
validation_pred = district_cases.shift(freq="7D").reindex(validation_weeks)
validation_actual = district_cases.loc[validation_weeks]

# Inspect one district so we can check the calculation by hand.
colombo_example = pd.DataFrame({
    "predicted_cases": validation_pred["Colombo"],
    "actual_cases": validation_actual["Colombo"],
})

colombo_example["absolute_error"] = (
    colombo_example["actual_cases"] - colombo_example["predicted_cases"]
).abs()

display(colombo_example.head())

,predicted_cases,actual_cases,absolute_error
week_start,,,
2024-09-14,226.0,202.0,24.0
2024-09-21,202.0,143.0,59.0
2024-09-28,143.0,169.0,26.0
2024-10-05,169.0,172.0,3.0
2024-10-12,172.0,180.0,8.0


In [8]:
absolute_errors = (validation_actual - validation_pred).abs()

# One error for each district-week with both a prediction and an actual count.
scored_errors = absolute_errors.stack()
baseline_mae = scored_errors.mean()

print("District-week predictions scored:", len(scored_errors))
print(f"Baseline validation MAE: {baseline_mae:.2f} cases")

district_results = pd.DataFrame({
    "weeks_scored": absolute_errors.count(),
    "mae_cases": absolute_errors.mean(),
})

display(district_results.sort_values("mae_cases", ascending=False).round(2))

District-week predictions scored: 1300
Baseline validation MAE: 9.00 cases


,weeks_scored,mae_cases
district,,
Colombo,52,31.56
Gampaha,52,21.85
Ratnapura,52,19.40
Kandy,52,16.85
Kalutara,52,12.71
Galle,52,12.25
Matara,52,11.15
Kegalle,52,9.50
Kurunegala,52,9.02


The baseline produced 1,300 validation predictions: 52 weeks for each of
the 25 districts. Across those predictions, the forecast differed from the
reported count by an average of 9.00 cases per district-week.

Colombo's average error was 31.56 cases, which is larger than the overall
average. This tells me the size of the error varies by district.